# Title

In [4]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
%reset -f

import numpy as np

n, d  = 10_000_000, 10
A = np.random.randn(d, d)
x = np.random.randn(n, d)

def f(x):
    return np.einsum("...d, ...e, de -> ...", x, x, A, optimize=True)

def g(x):
    return (x @ A * x).sum(len(x.shape) - 1)

assert f(x).shape == g(x).shape
np.allclose(f(x), g(x))

True

In [3]:
%%timeit
f(x)

305 ms ± 4.58 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
%%timeit
g(x)

358 ms ± 3.99 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [1]:
%reset -f

import mxnet as mx
from mxnet import np

n, d  = 10_000_000, 10
A = np.random.normal(0, 1, size=(d, d), ctx=mx.gpu())
x = np.random.normal(0, 1, size=(n, d), ctx=mx.gpu())

def f(x):
    return np.einsum("...d, ...e, de -> ...", x, x, A, optimize=True)

def g(x):
    return (x @ A * x).sum(len(x.shape) - 1)

assert f(x).shape == g(x).shape
np.allclose(f(x), g(x))

[11:35:52] ../src/base.cc:80: cuDNN lib mismatch: linked-against version 8202 != compiled-against version 8004.  Set MXNET_CUDNN_LIB_CHECKING=0 to quiet this warning.
[11:35:54] ../src/storage/storage.cc:199: Using Pooled (Naive) StorageManager for GPU


False

In [2]:
%%timeit
f(x).wait_to_read()

21.2 ms ± 41 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [3]:
%%timeit
g(x).wait_to_read()

4.43 ms ± 20 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [1]:
%reset -f
import jax
# jax.config.update('jax_platform_name', 'cpu')
print(jax.devices())
import jax.numpy as np
from jax import random, jit
key = random.PRNGKey(0)

n, d  = 10_000_000, 10
A = jax.random.normal(key, shape=(d, d))
x = jax.random.normal(key, shape=(n, d))

@jit
def f(x):
    return np.einsum("...d, ...e, de -> ...", x, x, A, optimize=True)
@jit
def g(x):
    return (x @ A * x).sum(len(x.shape) - 1)

assert f(x).shape == g(x).shape
np.allclose(f(x), g(x))

[GpuDevice(id=0, process_index=0)]


DeviceArray(True, dtype=bool)

In [2]:
%%timeit
f(x).block_until_ready()

2.45 ms ± 16 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [3]:
%%timeit
g(x).block_until_ready()

2.45 ms ± 11.3 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [1]:
%reset -f
import cupy as np

n, d  = 10_000_000, 10
A = np.random.randn(d, d)
x = np.random.randn(n, d)

def f(x):
    return np.einsum("...d, ...e, de -> ...", x, x, A, optimize=True)

def g(x):
    return (x @ A * x).sum(len(x.shape) - 1)

assert f(x).shape == g(x).shape
np.allclose(f(x), g(x))

array(True)

In [2]:
%%timeit
f(x)

76.5 ms ± 925 ns per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [3]:
%%timeit
g(x)

29.1 ms ± 23.5 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [6]:
%reset -f

import torch
from torch import jit, Tensor

n, d  = 10_000_000, 10
A = torch.randn(d, d, device="cuda")
x = torch.randn(n, d, device="cuda")

@jit.script
def f(A: Tensor, x: Tensor) -> Tensor:
    return torch.einsum("...d, ...e, de -> ...", x, x, A)

@jit.script
def g(A: Tensor, x: Tensor):
    return (x @ A * x).sum(len(x.shape) - 1)

assert f(A, x).shape == g(A, x).shape
torch.allclose(f(A, x), g(A, x))

False

In [7]:
%%timeit
f(A, x)

16.3 ms ± 35.6 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [8]:
%%timeit
g(A, x)

3.7 ms ± 6.98 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
